# धडा ०१ - AI एजंटसाठी परिचय

**AI एजंटसाठी सुरुवातीसाठी** कोर्समधील पहिल्या धड्याद तुम्हाचे स्वागत आहे!

एक **AI एजंट** हा एक प्रोग्राम आहे जो मोठ्या लँग्वेज मॉडेल (LLM) चा वापर त्याच्या कारणात्मक इंजिन म्हणून करतो आणि वास्तविक जगात *क्रिया* करू शकतो — API कॉल करणे, डेटाबेस क्वेरी करणे, किंवा कोड चालवणे — वापरकर्त्याच्या वतीने एखादे उद्दिष्ट साध्य करण्यासाठी.

या नोटबुकमध्ये तुम्ही तुमचा पहिला एजंट तयार कराल: एक **ट्रॅव्हल एजंट** जो सुट्टीसाठी स्थळांचे शिफारस करेल. मार्गदर्शन करताना तुम्हाला हे शिकायला मिळेल:

१. **Microsoft Agent Framework** चा वापर करून Microsoft Foundry Agent Service शी कनेक्ट होणे.
२. एजंटला एक **टूल** देणे — एक सोपा Python फंक्शन ज्याला एजंट कॉल करू शकतो.
३. एजंट चालवणे आणि त्याच्या प्रतिक्रियेचे परीक्षण करणे.
४. एजंटच्या प्रतिक्रियेचा टोकन-टोकनप्रमाणे स्ट्रीमिंग करणे.


## सेटअप

हा नोटबुक चालवण्यापूर्वी, सुनिश्चित करा की तुमच्याकडे:

1. **मायक्रोसॉफ्ट फाउंड्री प्रोजेक्ट** ज्यामध्ये एक तैनात चॅट मॉडेल आहे (उदा. `gpt-5-mini`).
2. **Azure CLI वापरून लॉग इन केलेले आहे** — तुमच्या टर्मिनलमध्ये `az login` चालवा.
3. **आवश्यक पर्यावरण चल (environment variables) सेट केलेले आहेत:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तुमच्या मायक्रोसॉफ्ट फाउंड्री प्रोजेक्टचा एंडपॉइंट.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तैनात केलेल्या मॉडेलचे नाव.

खालील सेल तुम्हाला आवश्यक असलेले Python पॅकेजेस इंस्टॉल करतो.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## आपला पहिला एजंट तयार करणे

एका एजंटला दोन गोष्टींची गरज असते:

- **सूचना** ज्या त्याला *तो कोण आहे* आणि *कसा वागत आहे* हे सांगतात (एक सिस्टम प्रॉम्प्ट).
- **साधने** — `@tool` ने सजवलेल्या Python फंक्शन्स, ज्यांचा एजंट माहिती मिळवण्यासाठी किंवा क्रिया करण्यासाठी कॉल करू शकतो.

खाली एक सोपी साधन व्याख्या केली आहे जी लोकप्रिय सुटकी स्थळांची यादी परत करते. वापरकर्ता प्रवास शिफारसी विचारल्यास एजंट ही साधन वापरेल.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिंग प्रतिसाद

अधिक संवादात्मक अनुभवासाठी तुम्ही एजंटचा प्रतिसाद **स्ट्रीम** करू शकता. पूर्ण उत्तराची वाट पाहण्याऐवजी, एजंट तयार होताच मजकूराचे तुकडे देतो. हे विशेषतः चॅट इंटरफेसमध्ये उपयुक्त आहे जिथे तुम्हाला आउटपुट रिअल टाइममध्ये दाखवायचा असतो.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

या धड्यात तुम्ही कसे शिकलात:

- **एक प्रदाता तयार करा** जो `FoundryChatClient` द्वारे Microsoft Foundry Agent Service शी कनेक्ट करतो.
- **@tool डेकोरेटर वापरून एक साधन परिभाषित करा जेणेकरून एजेंट तुमच्या Python फंक्शन्सना कॉल करू शकेल.**
- **वापरकर्ता संदेशासह एजेंट चालवा आणि त्याचे उत्तर छापा.**
- **प्रतिसाद रिअल-टाइम आउटपुटसाठी स्ट्रीम करा.**

पुढील धड्यात आपण एजेंटिक फ्रेमवर्क अधिक खोलवर पाहणार आहोत आणि एजेंट्सना अधिक सामर्थ्यशाली साधने आणि बहु-टप्प्याचे विश्लेषणात्मक क्षमताही कसे द्यायचे ते शिकणार आहोत.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
